# Module 02 — CNNs Deep Dive

We explore the key convolution variants that power modern efficient architectures:
receptive fields, depthwise separable convolutions, dilated convolutions,
transposed convolutions, group convolutions, and ASPP.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from layers import DepthwiseSeparableConv, DilatedConv, TransposedConvBlock, ASPPModule, receptive_field
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Receptive Field Calculator

The receptive field tells us how large a region in the input affects one output neuron.

In [ ]:
# 5 stacked 3x3 convolutions, stride 1
rf = receptive_field([3,3,3,3,3], [1,1,1,1,1])
print(f'5x 3x3 conv (stride 1): RF = {rf}')  # 11

# VGG-style: two 3x3 + pool
rf_vgg = receptive_field([3,3,2], [1,1,2])
print(f'2x 3x3 + 2x2 pool: RF = {rf_vgg}')

# With dilation
rf_dilated = receptive_field([3,3,3], [1,1,1], dilations=[1,2,4])
print(f'3x 3x3 with dilations [1,2,4]: RF = {rf_dilated}')

## 2. Depthwise Separable Convolutions

MobileNet's key insight: factorize standard conv into depthwise + pointwise.

In [ ]:
# Parameter comparison
in_c, out_c, k = 64, 128, 3
std_params = in_c * out_c * k * k
dws_params = in_c * k * k + in_c * out_c  # depthwise + pointwise
print(f'Standard conv params: {std_params:,}')
print(f'Depthwise separable:  {dws_params:,}')
print(f'Reduction factor: {std_params/dws_params:.2f}x')

# Forward pass
dws = DepthwiseSeparableConv(64, 128).to(DEVICE)
x = torch.randn(2, 64, 56, 56).to(DEVICE)
y = dws(x)
print('Output shape:', y.shape)

## 3. Dilated (Atrous) Convolutions

Dilation expands the receptive field without striding, preserving resolution.

In [ ]:
# Visualize effective receptive fields of dilated kernels
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, d in zip(axes, [1, 2, 4, 8]):
    grid = np.zeros((15, 15))
    eff_k = 3 + (3-1)*(d-1)
    start = (15 - eff_k) // 2
    for i in range(3):
        for j in range(3):
            ry, rx = start + i*d, start + j*d
            if 0 <= ry < 15 and 0 <= rx < 15:
                grid[ry, rx] = 1.0
    ax.imshow(grid, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Dilation={d}, RF={eff_k}x{eff_k}')
    ax.axis('off')
plt.suptitle('3x3 kernel at different dilation rates')
plt.tight_layout(); plt.show()

## 4. Transposed Convolutions

Transposed conv (often called 'deconv') learns to upsample feature maps.

In [ ]:
up2 = TransposedConvBlock(256, 128, scale_factor=2).to(DEVICE)
x_down = torch.randn(1, 256, 14, 14).to(DEVICE)
x_up = up2(x_down)
print(f'Input:  {x_down.shape}')  # (1, 256, 14, 14)
print(f'Output: {x_up.shape}')    # (1, 128, 28, 28)

## 5. ASPP Module

Atrous Spatial Pyramid Pooling aggregates multi-scale context from a feature map.

In [ ]:
aspp = ASPPModule(in_channels=512, out_channels=256, dilations=[1, 6, 12, 18]).to(DEVICE)
feat = torch.randn(1, 512, 32, 32).to(DEVICE)
out = aspp(feat)
print('ASPP output:', out.shape)  # (1, 256, 32, 32)

params = sum(p.numel() for p in aspp.parameters())
print(f'ASPP parameters: {params:,}')

## Exercise — Bottleneck with Dilation

Implement a `DilatedBottleneck` block that is identical to a standard `Bottleneck`
except its 3x3 conv uses `dilation=2` (and appropriate padding).

In [ ]:
### EXERCISE
class DilatedBottleneck(nn.Module):
    expansion = 4
    def __init__(self, in_planes, planes, stride=1, dilation=2):
        super().__init__()
        # TODO: implement
        raise NotImplementedError
    def forward(self, x):
        raise NotImplementedError